In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [ ]:
# 2. Load dataset
df = pd.read_csv("data/tweets.csv")

In [ ]:
# 3. Preprocessing
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

In [ ]:
df['clean_tweet'] = df['tweet'].apply(clean_text)
df.head()

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1,2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(df['clean_tweet'])
y = df['Sentiment']

In [ ]:
# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 6. Models
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM": SVC()
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    cv_score = cross_val_score(model, X, y, cv=10).mean()

    results.append([name, accuracy, precision, recall, f1, cv_score])

    print(f"\n{name}")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("CV Score:", cv_score)

In [ ]:
results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score", "CV Score"
])

results_df

In [ ]:
results_df.to_csv("results/model_results.csv", index=False)

In [ ]:
x = np.arange(len(results_df["Model"]))

plt.figure()
plt.bar(x, results_df["Accuracy"])
plt.xticks(x, results_df["Model"])
plt.title("Accuracy Comparison")

plt.savefig("results/accuracy_comparison.png")
plt.show()

In [ ]:
plt.figure()
plt.bar(x, results_df["Precision"])
plt.xticks(x, results_df["Model"])
plt.title("Precision Comparison")

plt.savefig("results/precision_comparison.png")
plt.show()

In [ ]:
plt.figure()
plt.bar(x, results_df["Recall"])
plt.xticks(x, results_df["Model"])
plt.title("Recall Comparison")

plt.savefig("results/recall_comparison.png")
plt.show()

In [ ]:
plt.figure()
plt.bar(x, results_df["F1 Score"])
plt.xticks(x, results_df["Model"])
plt.title("F1 Score Comparison")

plt.savefig("results/f1_comparison.png")
plt.show()

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm)

    plt.figure()
    disp.plot()
    plt.title(f"{name} Confusion Matrix")

    plt.savefig(f"results/{name}_cm.png")
    plt.show()